In [ ]:
!pip install transformers datasets accelerate -q

In [ ]:
import torch, math
from transformers import GPT2LMHeadModel, GPT2Tokenizer, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import Dataset

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2')

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
def generate_text(model, tokenizer, prompt):
    inputs = tokenizer.encode(prompt, return_tensors='pt')

    output = model.generate(
        inputs,
        max_length=80,
        temperature=0.8,
        top_k=50,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)

In [ ]:
prompts = [
    "This product is",
    "I bought this phone and",
    "The quality of this item"
]

print("=== BEFORE FINE-TUNING ===")
baseline = {}

for p in prompts:
    baseline[p] = generate_text(model, tokenizer, p)
    print(f"\nPrompt: {p}")
    print(f"Output: {baseline[p]}")

=== BEFORE FINE-TUNING ===

Prompt: This product is
Output: This product is gluten free and dairy free and made from scratch with no preservatives and no added sugar syrup.

I make this chicken breast tenderloin with fresh ginger garlic paste and turmeric chutney and cook on medium heat for five minutes until the chicken is tender.

When the chicken is ready to cook, add the chicken breasts and cook on medium heat for one more minutes

Prompt: I bought this phone and
Output: I bought this phone and my wife and we wanted something a little more refined and it really grabbed our attention.

Ingredients: soy sauce, sugar free flour and baking soda, kasuri methi and salt
Cook on low heat for three minutes until completely al dente. Remove from heat and stir in soy sauce. Serve quickly with steamed rice.
I love this post and hope

Prompt: The quality of this item
Output: The quality of this item is outstanding for the money and the price is worth every penny. We are happy customer has purch

In [ ]:
corpus = [
    'this phone has an amazing battery life and the camera quality is outstanding for the price.',
    'i bought this laptop for college and it handles all my assignments and coding projects perfectly.',
    'the sound quality of these headphones is incredible with deep bass and clear vocals.',
    'this smartwatch tracks my steps accurately and the heart rate monitor is very reliable.',
    'great wireless earbuds with noise cancellation that blocks out all background sound.',
    'the keyboard feels very comfortable for long typing sessions and the backlight is a nice touch.',
    'this portable charger saved me during travel and it charges my phone three times on a single charge.',
    'the tablet screen is bright and colorful which makes watching movies a great experience.',
    'i love this fitness tracker because it motivates me to reach my daily exercise goals.',
    'this bluetooth speaker is compact but delivers surprisingly loud and clear audio.',
    'the delivery was fast and the product was packed securely with no damage at all.',
    'excellent value for money and the build quality feels premium despite the affordable price.',
    'the customer service team was very helpful when i had questions about the product features.',
    'this camera takes stunning photos in low light and the video recording quality is very smooth.',
    'i have been using this product for three months and it still works perfectly like day one.',
    'the design is sleek and modern and it looks great on my desk next to my other gadgets.',
    'easy to set up right out of the box and the instructions were clear and simple to follow.',
    'highly recommend this product to anyone looking for quality and reliability at a fair price.',
    'the software updates keep adding new features which makes this purchase even more worthwhile.',
    'best purchase i made this year and i would definitely buy from this brand again.',
]

In [ ]:
dataset = Dataset.from_dict({'text': corpus})

def tokenize_function(example):
    return tokenizer(example['text'], truncation=True, padding='max_length', max_length=64)

tokenized = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

In [ ]:
split = tokenized.train_test_split(test_size=0.2)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    learning_rate=5e-5,
    logging_steps=10,
    save_strategy="no"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    data_collator=data_collator
)

In [ ]:
trainer.train()

Step,Training Loss
10,0.903781
20,0.571305
30,0.436381
40,0.413773


TrainOutput(global_step=40, training_loss=0.5813100457191467, metrics={'train_runtime': 127.9445, 'train_samples_per_second': 0.625, 'train_steps_per_second': 0.313, 'total_flos': 2612920320000.0, 'train_loss': 0.5813100457191467, 'epoch': 5.0})

In [ ]:
eval_result = trainer.evaluate()
print("Perplexity:", math.exp(eval_result["eval_loss"]))

Perplexity: 30.27015399692356


In [ ]:
print("\n=== AFTER FINE-TUNING ===")

for p in prompts:
    output = generate_text(model, tokenizer, p)

    print(f"\nPrompt: {p}")
    print(f"Before: {baseline[p]}")
    print(f"After:  {output}")


=== AFTER FINE-TUNING ===

Prompt: This product is
Before: This product is gluten free and dairy free and made from scratch with no preservatives and no added sugar syrup.

I make this chicken breast tenderloin with fresh ginger garlic paste and turmeric chutney and cook on medium heat for five minutes until the chicken is tender.

When the chicken is ready to cook, add the chicken breasts and cook on medium heat for one more minutes
After:  This product is packed with durable construction that holds up to extreme temperatures and the build quality feels premium despite the affordable price. The backlight is a nice touch and the backlight is a nice touch. The backlight is a nice touch and the backlight is a nice touch. The backlight is a nice touch and the backlight is a nice touch. The screen is a nice touch and the

Prompt: I bought this phone and
Before: I bought this phone and my wife and we wanted something a little more refined and it really grabbed our attention.

Ingredients: 

Recip generation


In [ ]:

# Text generation function
def generate_text(prompt):
    inputs = tokenizer.encode(prompt, return_tensors='pt')
    output = model.generate(
        inputs,
        max_length=100,
        temperature=0.8,
        top_k=50,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)

# Baseline output
prompts = [
    "To make butter chicken",
    "For pasta carbonara",
    "To prepare chocolate cake"
]

print("=== BEFORE FINE-TUNING ===")
baseline = {}
for p in prompts:
    baseline[p] = generate_text(p)
    print(f"\nPrompt: {p}")
    print(f"Output: {baseline[p]}")

# Recipe dataset
recipes = [
    'to make butter chicken start by marinating chicken pieces in yogurt with turmeric chili powder and garam masala for one hour.',
    'heat butter in a pan and fry onions until golden brown then add ginger garlic paste and cook for two minutes.',
    'add tomato puree and cook on low heat for ten minutes until the oil separates from the masala.',
    'add the marinated chicken and cook on medium heat for fifteen minutes until fully cooked.',
    'finish with fresh cream and kasuri methi and serve hot with naan or steamed rice.',
    'for pasta carbonara boil spaghetti in salted water until al dente and reserve half cup of pasta water.',
    'fry diced pancetta in olive oil until crispy and set aside.',
    'whisk together egg yolks parmesan cheese and black pepper in a bowl.',
    'toss the hot pasta with pancetta and remove from heat then quickly stir in the egg mixture.',
    'the residual heat will cook the eggs into a creamy sauce and serve immediately with extra parmesan.',
    'to prepare vegetable stir fry heat sesame oil in a wok on high heat.',
    'add sliced bell peppers broccoli florets and snap peas and toss for three minutes.',
    'pour in soy sauce oyster sauce and a pinch of sugar and stir well.',
    'add minced garlic and ginger and cook for one more minute until fragrant.',
    'serve the stir fry over steamed jasmine rice and garnish with sesame seeds.',
    'for chocolate chip cookies cream together butter and sugar until light and fluffy.',
    'beat in eggs one at a time then add vanilla extract and mix well.',
    'fold in flour baking soda and salt then gently stir in chocolate chips.',
    'scoop dough onto a baking sheet and bake at 180 degrees for twelve minutes until golden.',
    'let cookies cool on the tray for five minutes before transferring to a wire rack.',
]



# Tokenization
dataset = Dataset.from_dict({'text': recipes})

def tokenize(example):
    return tokenizer(example['text'], truncation=True, padding='max_length', max_length=64)

tokenized = dataset.map(tokenize, batched=True)

# Split
split = tokenized.train_test_split(test_size=0.2)

# Data collator
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Training arguments
training_args = TrainingArguments(
    output_dir="./recipe_model",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    learning_rate=5e-5,
    logging_steps=10,
    save_strategy="no"
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    data_collator=data_collator
)

# Train
trainer.train()

# Evaluation
eval_result = trainer.evaluate()
print("\nPerplexity:", math.exp(eval_result["eval_loss"]))

# After fine-tuning
print("\n=== AFTER FINE-TUNING ===")
for p in prompts:
    output = generate_text(p)
    print(f"\nPrompt: {p}")
    print(f"Before: {baseline[p]}")
    print(f"After:  {output}")

=== BEFORE FINE-TUNING ===

Prompt: To make butter chicken
Output: To make butter chicken breast with a light and fluffy color is very easy to make and the skin feels firm and soft even when you are cooking. The flavor of this chicken breast is very creamy and the chicken is very tender when cooked. I would definitely buy this product from this brand again!

Rated 5 out of 5 by Anonymous from Great quality for the price This chicken breast recipe tastes incredible with a light brown exterior and the skin feels soft and succulent when cooked. I would definitely buy from

Prompt: For pasta carbonara
Output: For pasta carbonara strips, which help with the texture and the consistency of the pasta, are a must. This will keep for a long time.

Cauliflower and cauliflower soup with green onions and garlic is a must try. I'm very pleased with the consistency of this soup and my veggies are very flavorful. The vegetables are very dense and the broth is very rich and creamy. The soup is easy to 

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Step,Training Loss
10,3.534423
20,2.504096
30,2.132867
40,1.843920



Perplexity: 15.262568214852008

=== AFTER FINE-TUNING ===

Prompt: To make butter chicken
Before: To make butter chicken breast with a light and fluffy color is very easy to make and the skin feels firm and soft even when you are cooking. The flavor of this chicken breast is very creamy and the chicken is very tender when cooked. I would definitely buy this product from this brand again!

Rated 5 out of 5 by Anonymous from Great quality for the price This chicken breast recipe tastes incredible with a light brown exterior and the skin feels soft and succulent when cooked. I would definitely buy from
After:  To make butter chicken broth with salted water and cook on medium heat until well blended. Sprinkle with soy sauce and cook until crispy. Add chicken stock and cook on medium heat for five minutes until softened. Remove from heat and stir in chicken stock.

Step by Step Photos

Preheat oven to 350 degrees. Pour chicken broth into pan and add chicken stock and cook on medium heat un